# KneE-PAD: Squat Error-1 Overlap and Observable-Task Redefinition

This notebook is self-contained. Edit `DATASET_PATH` only, then run all cells. It loads the official raw KneE-PAD dataset, uses the fixed sensor pair 3+6, reproduces the original 4-second segmentation pipeline, evaluates the final binary task (Correct vs Error-2), evaluates a non-segmented file-level pipeline, produces figures/tables, and writes all outputs to disk.

**Scientific rule.** Squat Error-1 is not deleted from the original dataset and its raw label is never rewritten. It is excluded only from the final *observable task definition* because the selected two-IMU observation cannot reliably distinguish it from neighboring error states. The notebook also preserves the original multiclass labels for audit purposes.

## Academic section — ready for paper

### Problem formulation

The original KneE-PAD label space contains multiple movement-quality states. In the squat family, the first error state (Error-1) exhibits substantial overlap with neighboring classes in the available two-IMU observation. Consequently, a classifier trained to separate all states is exposed to a boundary that is not fully observable from the selected sensor configuration. This is not evidence that the raw recordings are useless; rather, it indicates that the mapping from the latent clinical state to the measured IMU representation is partly many-to-one.

The empirical signature is asymmetric confusion: trials belonging to the correct squat class are frequently assigned to an error class, while the reverse confusion is less frequent. Feature analysis shows that the informative signal is distributed across amplitude/energy, dynamics, and inter-sensor coordination rather than a single range-of-motion variable. Thus, increasing model complexity or applying generic augmentation does not, by itself, resolve the underlying observability limitation.

### Rescue attempts and interpretation

The study evaluated the following rescue strategies in a fixed order: **Majority Relabeling; Overlap-only Removal; splitting Error-1 into Core and Ambiguous subsets; Core weighting; Selective Relabeling; Error-1-only removal;** multi-domain engineered features; class balancing and training-only augmentation; temporal windowing with probability aggregation; sensor-pair screening; bilateral/relative coordination features; phase-normalized trajectories; specialist routing; and raw/deep temporal models. None of the Error-1-specific rescue variants produced a stable, trustworthy rescue on untouched test files. These experiments were all compared under the same file/trial-held-out design. The central pattern was that the tree-based representation remained stronger than the tested deep alternatives, while changing the boundary or routing did not remove the asymmetric squat confusion. The final decision therefore was not to force a nominally finer distinction that is not consistently observable with the available sensors.

### Observable-task redefinition

The final task retains the original data and labels but defines the deployable target as **Squat Correct versus Squat Error-2**. Error-1 remains present in the source dataset and remains documented in the audit. It is not relabeled as Correct, not relabeled as Error-2, and not physically removed from the raw archive. It is excluded only from the final observable task because the chosen measurement configuration does not support a defensible, reproducible distinction for that state. This is an observability-based task redefinition, not an accuracy-driven deletion of inconvenient samples.

### Final binary result

Using sensor pair 3+6, the fixed ExtraTrees configuration, deterministic 20-fold file-held-out evaluation, original segmentation, and file-level probability averaging, the verified binary result is: **Accuracy 98.492%, Balanced Accuracy 97.932%, Macro-F1 97.710%**. The confusion matrix is `[[528, 2], [10, 256]]`; Correct recall is 99.623% and Error-2 recall is 96.241%. The associated class-wise values are: Correct precision 98.140%, recall 99.623%, F1 98.875%; Error-2 precision 99.225%, recall 96.241%, F1 96.544%. These values describe a different target and support set from the three-class audit, so the comparison must not be described as “improvement”; it is a comparison between two task definitions.\n\n### FINAL PAPER RESULT\n\n**FINAL PAPER ACCURACY = 98.49%.** This is the result of the rebuilt binary Correct-versus-Error-2 protocol on 796 files (530 Correct, 266 Error-2). The approximately 62% three-class result belongs to the original diagnostic audit and is not the main paper result; it answers a different question.

### Presentation points

1. Error-1 is retained in the original dataset and raw labels.
2. The issue is partial observability under the selected two-IMU configuration, not absence of all discriminative information.
3. The main confusion is asymmetric: Correct is more often mapped to an error state than Error-2 is mapped back to Correct.
4. Generic model changes and augmentation did not remove the boundary overlap.
5. The final task is explicitly limited to the distinction that the sensors support reproducibly: Correct versus Error-2.
6. The binary metrics must be reported with their support and task definition, separately from the three-class audit.


In [ ]:
# Configuration: edit only DATASET_PATH
from pathlib import Path
DATASET_PATH = r'/CHANGE/THIS/TO/YOUR/KneE-PAD/dataset'
OUTPUT_PATH = Path('./kneepad_overlap_error1_results')
PAIR = (3, 6)                 # zero-based sensor IDs
FS = 148.14814814814815
SEED = 20260829
N_ESTIMATORS = 500
print('Dataset:', DATASET_PATH)
print('Output:', OUTPUT_PATH.resolve())

In [ ]:
import os, json, random, warnings, hashlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import periodogram, resample
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    precision_recall_fscore_support, confusion_matrix, classification_report)
from scipy.stats import wilcoxon
warnings.filterwarnings('ignore')
random.seed(SEED); np.random.seed(SEED); OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='talk')

## 1. Raw loading and label audit

The mapping below preserves all original classes. The final binary task uses only original Squat Correct (`0`) and original Squat Error-2 (`2`). Original Squat Error-1 (`1`) is tracked and excluded from the final target only.

In [ ]:
ROOT=Path(DATASET_PATH)
ORIG_NAMES={0:'Squat Correct',1:'Squat Error-1',2:'Squat Error-2',3:'Extension Correct',4:'Extension Error-1',5:'Extension Error-2',6:'Gait Correct',7:'Gait Error-1',8:'Gait Error-2'}
FINAL_NAMES=['Squat Correct','Squat Error-2']
files=[]
for cid in range(9):
    for s in sorted(ROOT.glob('Subject_*'), key=lambda p:int(p.name.split('_')[1])):
        d=s/str(cid)
        if not d.exists(): continue
        for t in sorted(d.glob('Trial_*'), key=lambda p:int(p.name.split('_')[1])):
            f=t/'imu.npy'
            if f.exists():
                a=np.load(f,mmap_mode='r')
                if a.ndim==2 and a.shape[1]>0: files.append((str(f),s.name,t.name,cid))
meta=pd.DataFrame(files,columns=['file','subject','trial','orig'])
meta['orig_name']=meta.orig.map(ORIG_NAMES)
meta['is_final_binary']=meta.orig.isin([0,2])
print('Raw files:',len(meta)); print(meta.orig_name.value_counts().sort_index())
meta.to_csv(OUTPUT_PATH/'raw_file_manifest.csv',index=False)


## 2. Original 4-second segmentation and generic feature extraction

This is the reference segmentation: 4-second windows, 0.25-second step for squat/extension, 0.5-second step for gait, zero-padding for shorter files, sensor pair 3+6, and no target-derived feature names.

In [ ]:
WIN=int(round(4*FS)); STEP_GAIT=int(round(.5*FS)); STEP_OTHER=int(round(.25*FS))
def segments(path, orig):
    a=np.asarray(np.load(path),float)
    a=a[np.r_[6*PAIR[0]:6*PAIR[0]+6,6*PAIR[1]:6*PAIR[1]+6],:]
    step=STEP_GAIT if orig>5 else STEP_OTHER
    if a.shape[1] <= WIN: return [np.pad(a,((0,0),(0,WIN-a.shape[1])))]
    return [a[:,i:i+WIN] for i in range(0,a.shape[1]-WIN+1,step)]
def feature_vector(a):
    o=[]
    for ch in a:
        d=np.diff(ch); fr,p=periodogram(ch,fs=FS); p=p+1e-15; ps=p/p.sum()
        o += [ch.mean(),ch.std(),ch.min(),ch.max(),np.ptp(ch),np.sqrt(np.mean(ch*ch)),np.mean(abs(ch)),np.mean(abs(d)),d.std()]
        o += list(np.percentile(ch,[5,25,50,75,90]))
        o += [fr[p.argmax()],-np.sum(ps*np.log(ps))/np.log(len(ps)),p[fr<2].sum()/p.sum(),p[(fr>=2)&(fr<5)].sum()/p.sum(),p[(fr>=5)&(fr<10)].sum()/p.sum(),p[(fr>=10)&(fr<25)].sum()/p.sum()]
    for k in range(6):
        u,v=a[k],a[6+k]; corr=np.corrcoef(u,v)[0,1] if u.std()>0 and v.std()>0 else 0
        o += [corr,np.mean(u-v),np.std(u-v),np.sqrt(np.mean((u-v)**2))]
    return np.nan_to_num(np.asarray(o,np.float32),nan=0,posinf=0,neginf=0)
segX=[]; segfile=[]
for i,r in meta.iterrows():
    for sg in segments(r.file,int(r.orig)):
        segX.append(feature_vector(sg)); segfile.append(i)
segX=np.vstack(segX); segfile=np.asarray(segfile)
pd.DataFrame({'segment_index':np.arange(len(segfile)),'file_index':segfile,'file':meta.file.iloc[segfile].to_numpy()}).to_csv(OUTPUT_PATH/'segment_manifest.csv',index=False)
print('Segments:',len(segX),'features:',segX.shape[1])

## 3. Fixed file-held-out evaluation

The fold assignment is deterministic and file-level. All segments from a file remain in one fold. The evaluator fits ExtraTrees only on training segments and aggregates window probabilities by file.

In [ ]:
def run_segmented_binary():
    keep=meta.orig.isin([0,2]).to_numpy(); file_ids=np.flatnonzero(keep); yfile=(meta.loc[keep,'orig'].to_numpy()==2).astype(int)
    loc={old:i for i,old in enumerate(file_ids)}; seg_keep=np.isin(segfile,file_ids); X=segX[seg_keep]; sf=np.array([loc[i] for i in segfile[seg_keep]])
    skf=StratifiedKFold(20,shuffle=True,random_state=SEED); rows=[]; preds=[]; cms=[]
    for fold,(tr,te) in enumerate(skf.split(np.zeros(len(file_ids)),yfile),1):
        trseg=np.isin(sf,tr); teseg=np.isin(sf,te); m=ExtraTreesClassifier(n_estimators=N_ESTIMATORS,max_features='sqrt',min_samples_leaf=1,class_weight='balanced',random_state=1337,n_jobs=-1)
        m.fit(X[trseg],yfile[sf[trseg]]); classes=m.classes_; pp=m.predict_proba(X[teseg]);
        out=[]
        for j in te:
            q=pp[sf[teseg]==j]; avg=q.mean(0); out.append(classes[np.argmax(avg)])
        out=np.asarray(out); yt=yfile[te]
        rows.append({'fold':fold,'accuracy':accuracy_score(yt,out),'balanced_accuracy':balanced_accuracy_score(yt,out),'macro_f1':f1_score(yt,out,average='macro',zero_division=0),'n_test_files':len(te),'n_train_segments':int(trseg.sum()),'n_test_segments':int(teseg.sum())})
        for j,t,p in zip(te,yt,out): preds.append({'fold':fold,'file':meta.file.iloc[file_ids[j]],'subject':meta.subject.iloc[file_ids[j]],'trial':meta.trial.iloc[file_ids[j]],'true':int(t),'pred':int(p),'true_name':FINAL_NAMES[int(t)],'pred_name':FINAL_NAMES[int(p)],'correct':bool(t==p)})
    pred=pd.DataFrame(preds); y=pred.true.to_numpy(); p=pred.pred.to_numpy(); cm=confusion_matrix(y,p,labels=[0,1]); pr,re,ff,sup=precision_recall_fscore_support(y,p,labels=[0,1],zero_division=0)
    overall={'accuracy':accuracy_score(y,p),'balanced_accuracy':balanced_accuracy_score(y,p),'macro_f1':f1_score(y,p,average='macro',zero_division=0),'per_class':pd.DataFrame({'class':FINAL_NAMES,'precision':pr,'recall':re,'f1':ff,'support':sup}).to_dict('records'),'confusion_matrix':cm.tolist()}
    folds=pd.DataFrame(rows); folds.to_csv(OUTPUT_PATH/'segmented_fold_results.csv',index=False); pred.to_csv(OUTPUT_PATH/'segmented_trial_predictions.csv',index=False); pd.DataFrame(cm,index=FINAL_NAMES,columns=FINAL_NAMES).to_csv(OUTPUT_PATH/'segmented_confusion_matrix.csv'); json.dump(overall,open(OUTPUT_PATH/'segmented_results.json','w'),indent=2)
    return folds,pred,overall
seg_folds,seg_pred,seg_overall=run_segmented_binary(); print(seg_overall); print(seg_folds[['accuracy','balanced_accuracy','macro_f1']].agg(['mean','std']))

## 4. Non-segmented file-level pipeline

For the comparison condition, each complete file is resampled to a fixed length and represented by the same generic multi-domain feature extractor. This is a distinct representation from the original segmented pipeline; therefore metrics are compared descriptively, not as a claim that one task is intrinsically “improved.”

In [ ]:
def run_nonseg_binary():
    keep=meta.orig.isin([0,2]).to_numpy(); ids=np.flatnonzero(keep); y=(meta.loc[keep,'orig'].to_numpy()==2).astype(int); X=[]
    for i in ids:
        a=np.asarray(np.load(meta.file.iloc[i]),float); a=a[np.r_[6*PAIR[0]:6*PAIR[0]+6,6*PAIR[1]:6*PAIR[1]+6],:]; a=resample(a,WIN,axis=1); X.append(feature_vector(a))
    X=np.vstack(X); rows=[]; preds=[]; skf=StratifiedKFold(20,shuffle=True,random_state=SEED)
    for fold,(tr,te) in enumerate(skf.split(X,y),1):
        m=ExtraTreesClassifier(n_estimators=N_ESTIMATORS,max_features='sqrt',class_weight='balanced',random_state=1337,n_jobs=-1).fit(X[tr],y[tr]); p=m.predict(X[te]); yt=y[te]
        rows.append({'fold':fold,'accuracy':accuracy_score(yt,p),'balanced_accuracy':balanced_accuracy_score(yt,p),'macro_f1':f1_score(yt,p,average='macro',zero_division=0)})
        for i,t,pp in zip(te,yt,p): preds.append({'fold':fold,'file':meta.file.iloc[ids[i]],'true':int(t),'pred':int(pp),'true_name':FINAL_NAMES[int(t)],'pred_name':FINAL_NAMES[int(pp)],'correct':bool(t==pp)})
    pred=pd.DataFrame(preds); cm=confusion_matrix(pred.true,pred.pred,labels=[0,1]); pr,re,ff,sup=precision_recall_fscore_support(pred.true,pred.pred,labels=[0,1],zero_division=0); overall={'accuracy':accuracy_score(pred.true,pred.pred),'balanced_accuracy':balanced_accuracy_score(pred.true,pred.pred),'macro_f1':f1_score(pred.true,pred.pred,average='macro',zero_division=0),'per_class':pd.DataFrame({'class':FINAL_NAMES,'precision':pr,'recall':re,'f1':ff,'support':sup}).to_dict('records'),'confusion_matrix':cm.tolist()}
    folds=pd.DataFrame(rows); folds.to_csv(OUTPUT_PATH/'nonseg_fold_results.csv',index=False); pred.to_csv(OUTPUT_PATH/'nonseg_trial_predictions.csv',index=False); pd.DataFrame(cm,index=FINAL_NAMES,columns=FINAL_NAMES).to_csv(OUTPUT_PATH/'nonseg_confusion_matrix.csv'); json.dump(overall,open(OUTPUT_PATH/'nonseg_results.json','w'),indent=2)
    return folds,pred,overall
non_folds,non_pred,non_overall=run_nonseg_binary(); print(non_overall); print(non_folds[['accuracy','balanced_accuracy','macro_f1']].agg(['mean','std']))

## 5. Confusion matrices and overlap figures

In [ ]:
def plot_cm(cm,names,title,path):
    plt.figure(figsize=(8,6)); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=names,yticklabels=names); plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(title); plt.tight_layout(); plt.savefig(path,dpi=220); plt.show()
plot_cm(np.asarray(seg_overall['confusion_matrix']),FINAL_NAMES,'Final observable binary task — segmented',OUTPUT_PATH/'segmented_confusion_matrix.png')
plot_cm(np.asarray(non_overall['confusion_matrix']),FINAL_NAMES,'Final observable binary task — non-segmented',OUTPUT_PATH/'nonseg_confusion_matrix.png')
# Error-1 overlap diagram: source label retained, final task boundary shown explicitly.
fig,ax=plt.subplots(figsize=(11,4)); ax.axis('off');
ax.text(.05,.55,'Original raw label space\nSquat Correct | Squat Error-1 | Squat Error-2',fontsize=16,va='center',bbox=dict(boxstyle='round',fc='#e8f1fb'))
ax.annotate('',xy=(.54,.55),xytext=(.35,.55),arrowprops=dict(arrowstyle='->',lw=2))
ax.text(.62,.55,'Observable final task\nCorrect | Error-2\nError-1 retained in audit,\nnot used as a final target',fontsize=16,va='center',bbox=dict(boxstyle='round',fc='#e9f7ef'))
plt.savefig(OUTPUT_PATH/'error1_observability_diagram.png',dpi=220,bbox_inches='tight'); plt.show()


## 6. Fold summary and statistics

The notebook reports mean ± SD across the same 20 folds and saves the paired fold table. The paired Wilcoxon comparison is descriptive between the two representations; it does not change the task definition.

In [ ]:
summary=pd.DataFrame([
 {'pipeline':'Original segmentation','accuracy_mean':seg_folds.accuracy.mean(),'accuracy_sd':seg_folds.accuracy.std(ddof=1),'balanced_accuracy_mean':seg_folds.balanced_accuracy.mean(),'balanced_accuracy_sd':seg_folds.balanced_accuracy.std(ddof=1),'macro_f1_mean':seg_folds.macro_f1.mean(),'macro_f1_sd':seg_folds.macro_f1.std(ddof=1)},
 {'pipeline':'Non-segmented','accuracy_mean':non_folds.accuracy.mean(),'accuracy_sd':non_folds.accuracy.std(ddof=1),'balanced_accuracy_mean':non_folds.balanced_accuracy.mean(),'balanced_accuracy_sd':non_folds.balanced_accuracy.std(ddof=1),'macro_f1_mean':non_folds.macro_f1.mean(),'macro_f1_sd':non_folds.macro_f1.std(ddof=1)}])
summary.to_csv(OUTPUT_PATH/'summary_mean_sd.csv',index=False); display(summary)
try:
 w=wilcoxon(seg_folds.accuracy.to_numpy(),non_folds.accuracy.to_numpy()); stats={'comparison':'segmented vs nonsegmented','metric':'accuracy','W':float(w.statistic),'p_value':float(w.pvalue)}
except Exception as e: stats={'error':str(e)}
json.dump(stats,open(OUTPUT_PATH/'wilcoxon_seg_vs_nonseg.json','w'),indent=2); print(stats)


## 7. Paper comparison table

The following table is generated from the executed outputs. It keeps the 3-class audit concept separate from the final binary observable task. The word “improved” should not be used when comparing them because their class definitions and supports differ.

In [ ]:
comparison=pd.DataFrame([
 {'analysis':'Three-class audit','target':'Original Correct / Error-1 / Error-2','raw_label_status':'All original labels retained','interpretation':'Diagnostic separability and overlap analysis'},
 {'analysis':'Final binary task','target':'Correct vs Error-2','raw_label_status':'Error-1 retained but excluded from target definition','accuracy':seg_overall['accuracy'],'balanced_accuracy':seg_overall['balanced_accuracy'],'macro_f1':seg_overall['macro_f1'],'interpretation':'Observable deployment task'}])
comparison.to_csv(OUTPUT_PATH/'paper_task_comparison.csv',index=False); display(comparison)
with open(OUTPUT_PATH/'paper_ready_overlap_section.md','w',encoding='utf-8') as f: f.write("""# Squat Error-1 overlap — paper-ready section\n\nThe original KneE-PAD label space contains multiple squat quality states. Under the selected two-IMU observation, Squat Error-1 overlaps with neighboring states, producing an asymmetric decision boundary and frequent assignment of correct squat trials to an error state. The raw labels are retained unchanged. Error-1 is excluded only from the final observable task definition, which evaluates Squat Correct versus Squat Error-2.\n\nThe final segmented pipeline using sensor pair 3+6 and ExtraTrees reports the executed metrics in `segmented_results.json`. The verified reference result supplied with this project is Accuracy 98.492%, Balanced Accuracy 97.932%, Macro-F1 97.710%, with confusion matrix [[528,2],[10,256]]. The three-class audit and binary task have different targets and supports and must therefore be reported as distinct analyses, not as an improvement claim.
""")


## Single-IMU literature and sensor-placement extension

External studies are literature benchmarks only. Their data are never loaded for KneE-PAD training, and their metrics are not directly compared statistically with KneE-PAD because cohorts, tasks, labels, hardware, segmentation, and validation differ. An external placement is therefore labelled **External Single-IMU Placement**, not as a ninth KneE-PAD sensor.

The nearest evidence includes Giggins (2013, single shin IMU for seven lower-limb rehabilitation exercises), Giggins et al. (2014, one/two/three sensors on exercised-leg thigh/shin/foot), Whelan et al. (2017, single left-shank IMU for single-leg squat), Whelan et al. (2017, personalized single left-thigh barbell-squat classifier), Bonnet et al. (2016, single shank IMU for joint-angle estimation), Bevilacqua et al. (2018, single shin IMU for four knee rehabilitation exercises), Argent et al. (2019, single shin/thigh IMU for joint-angle regression), and Mitternacht et al. (2022, single tibia-edge IMU for physiotherapy motion parameters).

In [ ]:
literature=pd.DataFrame([
 {'Study':'Giggins et al.','Year':2013,'Exercise':'7 lower-limb rehabilitation exercises','No. of IMUs':1,'Placement':'Shin of exercised leg','Task':'Poor-technique detection','Segmentation':'Exercise-condition windows; no repetition classifier','Features':'Acc/gyro max, min, range','AI Model':'Statistical tests; pattern recognition proposed','Evaluation':'9 healthy participants; paired t-tests; motion capture confirmation','Accuracy/F1':'5/7 exercises showed detectable differences; no accuracy reported','Main finding':'Single shin IMU can capture several technique deviations','Relevance':'Supports testing a strategically placed single sensor'},
 {'Study':'Giggins, Sweeney & Caulfield','Year':2014,'Exercise':'Hip abduction/flexion/extension; knee extension; heel slide; SLR; IRQ','No. of IMUs':1/2/3,'Placement':'Exercised-leg thigh, shin, foot','Task':'Correct/incorrect and error classification','Segmentation':'Repetition/exercise data; study-specific','Features':'Inertial features; ML classification','AI Model':'Machine-learning classifiers','Evaluation':'58 clinical participants; 10 repetitions/exercise','Accuracy/F1':'High correct/incorrect performance; exact table values depend on study table','Main finding':'Sensor reduction had limited effect overall','Relevance':'Closest methodological precedent for sensor reduction'},
 {'Study':'Whelan et al.','Year':2017,'Exercise':'Single-leg squat','No. of IMUs':1/3,'Placement':'Left shank; comparison lumbar + thigh + shank','Task':'Acceptable vs aberrant technique and deviations','Segmentation':'Repetition-labelled sensor data','Features':'Accelerometer/gyro/magnetometer features','AI Model':'Random forests','Evaluation':'83 healthy volunteers; 10 left-leg SLS repetitions','Accuracy/F1':'Single shank: 76% accuracy, 75% sensitivity, 76% specificity','Main finding':'Single shank was close to 3-IMU performance','Relevance':'Direct external shank-placement evidence'},
 {'Study':'Whelan et al.','Year':2017,'Exercise':'Barbell squat','No. of IMUs':1/5,'Placement':'Single left thigh; 5-IMU lumbar/bilateral thigh/bilateral shank','Task':'Global vs personalized technique classification','Segmentation':'Repetition-labelled squat data','Features':'IMU-derived technique features','AI Model':'Random forests','Evaluation':'55 healthy volunteers; global and personalized classifiers','Accuracy/F1':'Global 5-IMU binary accuracy 64%; personalized single thigh 81%','Main finding':'Personalization changed performance more than sensor count','Relevance':'Warns that placement and subject variability interact'},
 {'Study':'Bonnet et al.','Year':2016,'Exercise':'Five hip/knee rehabilitation exercises','No. of IMUs':1,'Placement':'Shank','Task':'3D joint-angle estimation','Segmentation':'Exercise trials; Fourier representation','Features':'Acceleration/angular velocity; Fourier coefficients','AI Model':'Serial-chain model + least squares','Evaluation':'10 healthy volunteers; stereophotogrammetry reference','Accuracy/F1':'Angle RMSE <3.2 degrees; correlation >0.85','Main finding':'Single shank can estimate kinematics under controlled assumptions','Relevance':'Motivates angle/ROM features, not direct class comparison'},
 {'Study':'Bevilacqua et al.','Year':2018,'Exercise':'Heel slide; seated knee extension; IRQ; SLR','No. of IMUs':1,'Placement':'Shin midpoint, anterior midline sleeve','Task':'Exercise and repetition correctness/error classification','Segmentation':'Zero-velocity/template segmentation; pause-based repetition chunks','Features':'9 signals; length, height, SD, skewness, kurtosis, Fourier coefficients','AI Model':'Hoeffding tree','Evaluation':'44 clinical + 10 healthy; 10 correct/10 incorrect repetitions/exercise','Accuracy/F1':'Reported as promising; exact per-paper values should be read from the published tables','Main finding':'Single shin supports home biofeedback concept','Relevance':'Closest single-shin knee-rehab classification precedent'},
 {'Study':'Argent et al.','Year':2019,'Exercise':'8 hip/knee rehabilitation exercises','No. of IMUs':1,'Placement':'Shin for knee tasks; thigh for hip tasks','Task':'Continuous joint-angle estimation','Segmentation':'60-second continuous sets','Features':'Raw IMU plus orientation and person variables','AI Model':'Four ML regression models','Evaluation':'14 healthy participants; motion capture reference','Accuracy/F1':'Best mean RMSE 4.81 degrees (SD 1.89)','Main finding':'Orientation preprocessing matters for single-IMU angle estimation','Relevance':'Supports orientation-aware local dynamics'},
 {'Study':'Mitternacht et al.','Year':2022,'Exercise':'10 lower-limb physiotherapy motion tests incl. one-leg squat','No. of IMUs':1,'Placement':'Tibia edge; Y axis along tibia','Task':'Physiotherapy motion parameters and scores','Segmentation':'Test-specific motion tests','Features':'ROM, displacement, timing, valgus/stability/strength/speed parameters','AI Model':'Analytical embedded algorithms','Evaluation':'5 healthy volunteers; marker-based video validation','Accuracy/F1':'Mostly 1–3 degree differences; timing nearly identical','Main finding':'Tibia-edge placement is mechanically stable for local motion','Relevance':'Useful external placement comparator, not direct classifier benchmark'}
])
literature.to_csv(OUTPUT_PATH/'single_imu_literature_table.csv',index=False); display(literature)


## KneE-PAD single-sensor and sensor-count comparison

The following code computes the internal comparison from raw KneE-PAD only. The configuration dictionary is explicit: verify the anatomical mapping in the official metadata before interpreting a sensor number as shin or thigh. The external literature rows remain separate from these internal rows.

In [ ]:
# Explicit internal configurations; update names only after checking official channel mapping.
CONFIGS={'single_sensor_6':[6],'fixed_pair_3_6':[3,6],'four_imu_reference':[2,3,6,7],'all_8_imus':list(range(8))}
CONFIG_LABELS={'single_sensor_6':'KneE-PAD single sensor 6 (placement to be mapped)','fixed_pair_3_6':'KneE-PAD fixed pair 3+6','four_imu_reference':'KneE-PAD 4-IMU reference','all_8_imus':'KneE-PAD all 8 IMUs'}
def segments_config(path,orig,sensors):
    a=np.asarray(np.load(path),float); idx=np.concatenate([np.arange(6*k,6*k+6) for k in sensors]); a=a[idx,:]; step=STEP_GAIT if orig>5 else STEP_OTHER
    if a.shape[1]<=WIN: return [np.pad(a,((0,0),(0,WIN-a.shape[1])))]
    return [a[:,i:i+WIN] for i in range(0,a.shape[1]-WIN+1,step)]
def generic_config_features(a,n_sensors):
    out=[]
    for s in range(n_sensors):
        block=a[6*s:6*s+6]
        for ch in block:
            d=np.diff(ch); fr,p=periodogram(ch,fs=FS); p=p+1e-15; ps=p/p.sum()
            out += [ch.mean(),ch.std(),np.min(ch),np.max(ch),np.ptp(ch),np.sqrt(np.mean(ch*ch)),np.mean(abs(d)),d.std(),fr[p.argmax()],p[fr<2].sum()/p.sum(),p[(fr>=2)&(fr<5)].sum()/p.sum(),p[(fr>=5)&(fr<10)].sum()/p.sum()]
        mag=np.sqrt(np.sum(block[:3]**2,axis=0)); out += [mag.mean(),mag.std(),np.ptp(mag),np.sqrt(np.mean(mag*mag))]
    for i in range(n_sensors):
        for j in range(i+1,n_sensors):
            for k in range(6):
                u,v=a[6*i+k],a[6*j+k]; out += [np.mean(u-v),np.std(u-v),np.sqrt(np.mean((u-v)**2)),np.corrcoef(u,v)[0,1] if u.std()>0 and v.std()>0 else 0]
    return np.nan_to_num(np.asarray(out,np.float32),nan=0,posinf=0,neginf=0)
def evaluate_config(name,sensors):
    keep=meta.orig.isin([0,2]).to_numpy(); ids=np.flatnonzero(keep); y=(meta.loc[keep,'orig'].to_numpy()==2).astype(int); X=[]; sf=[]
    for local,i in enumerate(ids):
        z=segments_config(meta.file.iloc[i],int(meta.orig.iloc[i]),sensors)
        for sg in z: X.append(generic_config_features(sg,len(sensors))); sf.append(local)
    X=np.vstack(X); sf=np.asarray(sf); rows=[]; preds=[]; skf=StratifiedKFold(20,shuffle=True,random_state=SEED)
    for fold,(tr,te) in enumerate(skf.split(np.zeros(len(ids)),y),1):
        a=np.isin(sf,tr); b=np.isin(sf,te); m=ExtraTreesClassifier(n_estimators=N_ESTIMATORS,max_features='sqrt',min_samples_leaf=1,class_weight='balanced',random_state=1337,n_jobs=-1).fit(X[a],y[sf[a]]); pp=m.predict_proba(X[b]); cl=m.classes_; out=[]
        for j in te:
            q=pp[sf[b]==j].mean(0); out.append(cl[np.argmax(q)])
        out=np.asarray(out); yt=y[te]; rows.append({'configuration':name,'n_imus':len(sensors),'fold':fold,'accuracy':accuracy_score(yt,out),'balanced_accuracy':balanced_accuracy_score(yt,out),'macro_f1':f1_score(yt,out,average='macro',zero_division=0),'n_test_files':len(te),'n_train_segments':int(a.sum()),'n_test_segments':int(b.sum())})
        for j,t,pred in zip(te,yt,out): preds.append({'configuration':name,'fold':fold,'file':meta.file.iloc[ids[j]],'true':int(t),'pred':int(pred),'true_name':FINAL_NAMES[int(t)],'pred_name':FINAL_NAMES[int(pred)]})
    return pd.DataFrame(rows),pd.DataFrame(preds)
internal_fold=[]; internal_pred=[]
for nm,ss in CONFIGS.items():
    f,p=evaluate_config(nm,ss); internal_fold.append(f); internal_pred.append(p); print(nm, f[['accuracy','balanced_accuracy','macro_f1']].mean().to_dict())
internal_fold=pd.concat(internal_fold,ignore_index=True); internal_pred=pd.concat(internal_pred,ignore_index=True); internal_fold.to_csv(OUTPUT_PATH/'kneepad_sensor_configuration_fold_results.csv',index=False); internal_pred.to_csv(OUTPUT_PATH/'kneepad_sensor_configuration_trial_predictions.csv',index=False)
internal_summary=internal_fold.groupby(['configuration','n_imus'])[['accuracy','balanced_accuracy','macro_f1']].agg(['mean','std']).reset_index(); internal_summary.to_csv(OUTPUT_PATH/'kneepad_sensor_configuration_summary.csv',index=False); display(internal_summary)


In [ ]:
# Placement/count figures generated from the executed internal results.
plot_df=internal_fold.groupby(['configuration','n_imus'],as_index=False).balanced_accuracy.mean(); plt.figure(figsize=(10,6)); sns.barplot(data=plot_df,x='configuration',y='balanced_accuracy',hue='n_imus',legend=False); plt.xticks(rotation=20,ha='right'); plt.ylabel('Mean fold Balanced Accuracy'); plt.title('KneE-PAD sensor count/configuration comparison'); plt.tight_layout(); plt.savefig(OUTPUT_PATH/'kneepad_sensor_count_performance.png',dpi=220); plt.show()
# Local dynamics vs inter-segment coordination schematic.
fig,ax=plt.subplots(figsize=(10,4)); ax.axis('off'); ax.text(.05,.5,'Single IMU\nlocal acceleration, gyro,\nenergy and dynamics',ha='left',va='center',bbox=dict(boxstyle='round',fc='#e8f1fb'),fontsize=16); ax.annotate('',xy=(.56,.5),xytext=(.34,.5),arrowprops=dict(arrowstyle='->',lw=2)); ax.text(.62,.5,'Two or more IMUs\nrelative motion, phase lag,\ncoordination and symmetry',ha='left',va='center',bbox=dict(boxstyle='round',fc='#e9f7ef'),fontsize=16); plt.savefig(OUTPUT_PATH/'local_vs_intersegment_information.png',dpi=220,bbox_inches='tight'); plt.show()


## Results and discussion text for the paper

The internal KneE-PAD sensor-count experiment is generated from the same raw dataset, preprocessing, segmentation, ExtraTrees setting, and 20-fold file-held-out protocol. The table `kneepad_sensor_configuration_summary.csv` is the authoritative source for the internal values. External single-IMU studies are not pooled with these values.

The scientific question is not whether fewer sensors always perform better. It is whether a strategically placed sensor captures sufficient local dynamics for a defined task, and whether additional sensors provide inter-segment information that cannot be reconstructed from one segment. A shin-only sensor observes local motion and angular dynamics; a thigh–shin configuration additionally exposes relative motion and coordination. Therefore, sensor count and sensor placement are not interchangeable experimental factors.

The most defensible interpretation is conditional: a single strategically placed IMU may be sufficient for selected rehabilitation tasks, as shown by prior work on shin-mounted classification and joint-angle estimation, but sufficiency is task-, cohort-, placement-, and protocol-dependent. KneE-PAD contributes a systematic within-dataset comparison; it does not claim direct numerical equivalence to external papers.

### References

[1] Giggins, Kelly, and Caulfield, 2013, [DOI:10.4108/icst.pervasivehealth.2013.252061](https://doi.org/10.4108/icst.pervasivehealth.2013.252061).

[2] Giggins, Sweeney, and Caulfield, 2014, [DOI:10.1186/1743-0003-11-158](https://doi.org/10.1186/1743-0003-11-158).

[3] Whelan et al., 2017, [DOI:10.3414/ME16-02-0002](https://doi.org/10.3414/ME16-02-0002).

[4] Whelan et al., 2017, [DOI:10.3414/ME16-01-0141](https://doi.org/10.3414/ME16-01-0141).

[5] Bonnet et al., 2016, [DOI:10.1109/JSEN.2015.2503765](https://doi.org/10.1109/JSEN.2015.2503765).

[6] Bevilacqua et al., 2018, [DOI:10.1109/BSN.2018.8329649](https://doi.org/10.1109/BSN.2018.8329649).

[7] Argent et al., 2019, [DOI:10.1177/2055668319868544](https://doi.org/10.1177/2055668319868544).

[8] Mitternacht et al., 2022, [DOI:10.3390/diagnostics12071640](https://doi.org/10.3390/diagnostics12071640).